In [1]:
# The line below allows to print all the outputs of a cell instead of only the last one
# %config InteractiveShell.ast_node_interactivity = "all"
import os
import pathlib
import sys

import numpy as np
import pandas as pd
import yaml

In [5]:
from_script = True

print(f"WARNING: This notebook is set with from_script = {from_script}.")

if from_script:
    # Get the path of the notebook config file from the environment variable
    path_config_notebook = os.environ["PATH_YAML_CONFIG"]
    # Load the notebook config file
    with open(path_config_notebook, "r") as file:
        dict_config_notebook = yaml.safe_load(file)
    id_xp_dataset = dict_config_notebook["id_xp_dataset"]
    PATH_PROJECT = pathlib.Path(dict_config_notebook["path_project"])
    N_MAX_TRAJECTORIES = dict_config_notebook.get("n_max_trajectories", 20)
    OFFSET_TIME = dict_config_notebook.get("offset_time", 5)
    NAME_MLFLOW_STORE = dict_config_notebook.get(
        "name_mlflow_store", "mlruns_ruche_downloaded"
    )

    if "n_max_trajectories" not in dict_config_notebook:
        print(
            f"WARNING: N_MAX_TRAJECTORIES not in the config file."
            f"Default value is set to {N_MAX_TRAJECTORIES}"
        )
    if "offset_time" not in dict_config_notebook:
        print(
            f"WARNING: OFFSET_TIME not in the config file."
            f"Default value is set to {OFFSET_TIME}"
        )
else:
    id_xp_dataset = 21
    PATH_PROJECT = pathlib.Path(
        "/home/hosseinkhan/Documents/work/phd/git_repositories/control_dde"
    )
    N_MAX_TRAJECTORIES = 1000
    OFFSET_TIME = 5

    NAME_MLFLOW_STORE = "mlruns_ruche_downloaded"

# Add the path of the project to the sys.path in order to import
# the modules in the src folder
sys.path.insert(0, os.path.abspath(PATH_PROJECT))

# Define the paths
path_mlruns = pathlib.Path(f"{PATH_PROJECT}/data/{NAME_MLFLOW_STORE}")
path_experiment_dataset = pathlib.Path(f"{path_mlruns}/{id_xp_dataset}")
list_id_hash_dataset = [
    path.name
    for path in path_experiment_dataset.iterdir()
    if path.is_dir() and path.name != "tags"
]

print(f"Number of runs in the dataset: {len(list_id_hash_dataset)} \n")
print(f"XP id dataset: {id_xp_dataset} \n")

nested_dict_config_dataset = {}
list_df_config_flattened_dataset = []
nested_dict_trajectory_data = {}


def check_xp_lifecycle(path_file: pathlib.Path):
    if not path_file.exists():
        raise FileNotFoundError(f"The file {path_file} does not exist.")
    if not path_file.is_file():
        raise FileNotFoundError(f"The path {path_file} is not a file.")
    # Load the yaml file
    with open(path_file, "r") as _file:
        dict_temp = yaml.safe_load(_file)
    # Check the lifecycle key
    assert dict_temp.get("lifecycle_stage") == "active", "The experiment is not active."


path_xp_dataset = f"{path_mlruns}/{id_xp_dataset}"
for name_id in list_id_hash_dataset:
    print(f"ID: {name_id}")
    list_glob_config = list(
        pathlib.Path(f"{path_xp_dataset}/{name_id}/generated_data").glob(
            "./config/*.yaml"
        )
    )
    assert len(list_glob_config) != 0, "No config file in the directory."
    assert len(list_glob_config) == 1, "More than one config file in the directory."

    name_config_file = list_glob_config[0].name
    # Check if the file is a yaml file
    assert name_config_file.endswith(".yaml"), "File is not a yaml file."

    path_run_config_yaml = list_glob_config[0]
    with open(path_run_config_yaml, "r") as file:
        # Get the config file as a dictionary
        dict_config_temp = yaml.safe_load(file)
        # Add the config file to the nested dictionary
        nested_dict_config_dataset[name_id] = dict_config_temp
        # Flatten the config file for easier check
        list_df_config_flattened_dataset.append(
            pd.json_normalize(dict_config_temp, sep="_")
        )

    # Load numpy data from the trajectory_data folder
    path_run_trajectory_data = pathlib.Path(
        f"{path_xp_dataset}/{name_id}/generated_data/trajectory_data"
    )
    # Load the numpy data
    dict_trajectory_data = {}
    for path_file in path_run_trajectory_data.glob("*.npy"):
        if path_file.stem not in ["dones", "infos"]:
            dict_trajectory_data[path_file.stem] = np.load(path_file)
            print(
                f"Shape of {path_file.stem}:"
                f" {dict_trajectory_data[path_file.stem].shape}"
            )

    nested_dict_trajectory_data[name_id] = dict_trajectory_data

    del dict_config_temp

Number of runs in the dataset: 22 

XP id dataset: 21 

ID: 7b9425923c424262bb6fe6663bbc2e95
Shape of time: (40, 201)
Shape of truncated: (40, 200)
Shape of rewards: (40, 200)
Shape of observations: (40, 201, 1)
Shape of controls: (40, 200, 1)
ID: 012c4b89a6674b259c01f430b23e6dfd
Shape of time: (400, 201)
Shape of truncated: (400, 200)
Shape of rewards: (400, 200)
Shape of observations: (400, 201, 2)
Shape of controls: (400, 200, 1)
ID: ad0bd86499934c3ba07ec7de74988282
Shape of time: (400, 201)
Shape of truncated: (400, 200)
Shape of rewards: (400, 200)
Shape of observations: (400, 201, 2)
Shape of controls: (400, 200, 1)
ID: 43cefeefc71e4c368902a04b6cde7aa6
Shape of time: (40, 201)
Shape of truncated: (40, 200)
Shape of rewards: (40, 200)
Shape of observations: (40, 201, 1)
Shape of controls: (40, 200, 1)
ID: f48fb37d20ea48f3a3f7141bcf751363
Shape of time: (40, 201)
Shape of truncated: (40, 200)
Shape of rewards: (40, 200)
Shape of observations: (40, 201, 1)
Shape of controls: (40, 20

### DataFrames generation

#### DataFrame: Dataset config

In [9]:
df_dataset_config = (
    pd.concat(
        [
            df_config_flattened.assign(dataset_id=name_id)
            for name_id, df_config_flattened in zip(
                list_id_hash_dataset, list_df_config_flattened_dataset
            )
        ]
    )
    .set_index("dataset_id")
    .sort_values(["env_name", "env_params_max_control", "env_params_delay_observation"])
)
df_dataset_config.head(10)

,mlflow_experiment_name,seed,data_control_type,data_num_steps,data_num_trajectories,data_type,env_name,env_params_delay_observation,env_params_dict_solver_dt,env_params_dict_solver_name,env_params_dict_solver_order,env_params_dict_solver_stabilization,env_params_dt,env_params_log_callback_interval,env_params_max_control,env_params_mesh,env_params_name_flow,env_params_paraview_callback_interval,env_params_reynolds
dataset_id,,,,,,,,,,,,,,,,,,,
43cefeefc71e4c368902a04b6cde7aa6,2024_11_25_training_data_20h_58m,0,uniform,200,40,unique_trajectory_divided,cavity,0.0,0.001,semi_implicit_bdf,2,none,0.01,1000,0.00,coarse,cavity,1000,7500
f48fb37d20ea48f3a3f7141bcf751363,2024_11_25_training_data_20h_58m,0,uniform,200,40,unique_trajectory_divided,cavity,0.0,0.001,semi_implicit_bdf,2,none,0.01,1000,0.00,coarse,cavity,1000,5000
4e27a8e7a9c24b70acf9ba88d538cfcb,2024_11_25_training_data_20h_58m,0,uniform,200,40,unique_trajectory_divided,cavity,0.0,0.001,semi_implicit_bdf,2,none,0.01,1000,0.00,coarse,cavity,1000,500
7b9425923c424262bb6fe6663bbc2e95,2024_11_25_training_data_20h_58m,0,uniform,200,40,unique_trajectory_divided,cavity,0.0,0.001,semi_implicit_bdf,2,none,0.01,1000,0.01,coarse,cavity,1000,7500
a5b3d73e8ef34096ad8444bf8bbaa098,2024_11_25_training_data_20h_58m,0,uniform,200,40,unique_trajectory_divided,cavity,0.0,0.001,semi_implicit_bdf,2,none,0.01,1000,0.01,coarse,cavity,1000,500
e00c838399ad4f31be9ad1f3a33e26df,2024_11_25_training_data_20h_58m,0,uniform,200,40,unique_trajectory_divided,cavity,0.0,0.001,semi_implicit_bdf,2,none,0.01,1000,0.01,coarse,cavity,1000,5000
012c4b89a6674b259c01f430b23e6dfd,2024_11_25_training_data_20h_58m,0,uniform,200,400,unique_trajectory_divided,cylinder,0.0,0.010,semi_implicit_bdf,2,none,0.01,100,0.00,coarse,cylinder,100,90
ea32766566804a7e94de4ec18042b537,2024_11_25_training_data_20h_58m,0,uniform,200,400,unique_trajectory_divided,cylinder,0.0,0.010,semi_implicit_bdf,2,none,0.01,100,0.00,coarse,cylinder,100,105
cacb5c52a3d74da296747074fd5fb9b2,2024_11_25_training_data_20h_58m,0,uniform,200,400,unique_trajectory_divided,cylinder,0.0,0.010,semi_implicit_bdf,2,none,0.01,100,0.00,coarse,cylinder,100,120


## Plot Time Series

In [ ]:
# Plot the time series
import matplotlib.pyplot as plt


# Plot the observations for the first N_MAX_TRAJECTORIES trajectories
for i, (name_id, dict_trajectory_data) in enumerate(
    nested_dict_trajectory_data.items()
):
    dim_obs = dict_trajectory_data["observations"].shape[2]
    dim_control = dict_trajectory_data["controls"].shape[2]
    dim_reward = 1
    # Plot observations signal
    print(
        f"Observations for the first {N_MAX_TRAJECTORIES}"
        f" trajectories of the dataset {name_id}"
        f" with environment {df_dataset_config.loc[name_id]['env_name']}"
    )
    # Print the DataFrame entry associated with the dataset
    print(df_dataset_config.loc[name_id])
    fig, ax = plt.subplots(dim_obs + dim_control + dim_reward, 1, figsize=(16, 12))
    for j in range(dim_obs):
        ax[j].plot(
            dict_trajectory_data["time"][:N_MAX_TRAJECTORIES, OFFSET_TIME:].T,
            dict_trajectory_data["observations"][
                :N_MAX_TRAJECTORIES, OFFSET_TIME:, j
            ].T,
            # label=[f"Batch {i}" for i in range(N_MAX_TRAJECTORIES)],
        )
        ax[j].set_title(f"Obs y_{j}")
        ax[j].set_xlabel("Time - $t$")
        ax[j].set_ylabel("Obs - $y_{j}(t)$")
    for j in range(dim_control):
        ax[j + dim_obs].plot(
            dict_trajectory_data["time"][:N_MAX_TRAJECTORIES, OFFSET_TIME:-1].T,
            dict_trajectory_data["controls"][:N_MAX_TRAJECTORIES, OFFSET_TIME:, j].T,
            #             label=[f"Batch {i}" for i in range(N_MAX_TRAJECTORIES)],
        )
        ax[j + dim_obs].set_title(f"Control u_{j}")
        ax[j + dim_obs].set_xlabel("Time - $t$")
        ax[j + dim_obs].set_ylabel("Control - $u_{j}(t)$")
    # Plot rewards
    for j in range(dim_reward):
        ax[j + dim_obs + dim_control].plot(
            dict_trajectory_data["time"][:N_MAX_TRAJECTORIES, OFFSET_TIME + 1 :].T,
            dict_trajectory_data["rewards"][:N_MAX_TRAJECTORIES, OFFSET_TIME:].T,
            #             label=[f"Batch {i}" for i in range(N_MAX_TRAJECTORIES)],
        )
        ax[j + dim_obs + dim_control].set_title("Reward")
        ax[j + dim_obs + dim_control].set_xlabel("Time - $t$")
        ax[j + dim_obs + dim_control].set_ylabel("Reward - $r(t)$")

    fig.suptitle(
        f"Observations for the first {N_MAX_TRAJECTORIES}"
        f" trajectories of the dataset {name_id}"
    )
    fig.tight_layout()
    plt.show()
    del fig, ax